# Data Processing
Phase 1: Inspection | Phase 2: Cleaning | Phase 3: Feature Engineering

In [ ]:
import os
import io
import shutil
import pandas as pd
import numpy as np
import logging
from dotenv import load_dotenv

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

BASE_DIR = os.path.abspath('..')
DATA_DIR = os.path.join(BASE_DIR, 'data')
OUTPUT_DIR = os.path.join(BASE_DIR, 'outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load .env (gitignored) so KAGGLE_DATASET_ID is available
load_dotenv(os.path.join(BASE_DIR, '.env'))
print('Environment loaded.')

## Load Dataset (auto-download via kagglehub if needed)

In [ ]:
def get_data_path(data_dir):
    """Load CSV from local data/ folder, or download via kagglehub using KAGGLE_DATASET_ID from .env"""
    if os.path.exists(data_dir):
        csvs = [f for f in os.listdir(data_dir) if f.endswith('.csv')]
        if csvs:
            return os.path.join(data_dir, csvs[0])
    import kagglehub
    dataset_id = os.environ.get('KAGGLE_DATASET_ID')
    if not dataset_id:
        raise ValueError('KAGGLE_DATASET_ID not set. Add it to your .env file (see .env.example).')
    logging.info(f'Downloading dataset via kagglehub...')
    dl = kagglehub.dataset_download(dataset_id)
    csvs = [f for f in os.listdir(dl) if f.endswith('.csv')]
    os.makedirs(data_dir, exist_ok=True)
    dest = os.path.join(data_dir, csvs[0])
    shutil.copy(os.path.join(dl, csvs[0]), dest)
    return dest

data_path = get_data_path(DATA_DIR)
df_raw = pd.read_csv(data_path)
print(f'Loaded: {df_raw.shape}')
df_raw.head()

## Phase 1: Dataset Inspection

In [ ]:
print('Shape:', df_raw.shape)
print('\nColumn types:')
print(df_raw.dtypes)
print('\nMissing values:')
print(df_raw.isnull().sum())
print('\nDuplicate rows:', df_raw.duplicated().sum())
df_raw.describe(include='all')

In [ ]:
with open(os.path.join(OUTPUT_DIR, 'dataset_summary.txt'), 'w', encoding='utf-8') as f:
    f.write('=== Dataset Inspection Report ===\n\n')
    f.write(f'1. Shape: {df_raw.shape}\n\n')
    f.write('2. Head:\n' + df_raw.head().to_string() + '\n\n')
    buf = io.StringIO()
    df_raw.info(buf=buf)
    f.write('3. Info:\n' + buf.getvalue() + '\n')
    f.write('4. Describe:\n' + df_raw.describe(include='all').to_string() + '\n\n')
    f.write('5. Missing Values:\n' + df_raw.isnull().sum().to_string() + '\n\n')
    f.write(f'6. Duplicates: {df_raw.duplicated().sum()}\n')
print('Inspection report saved.')

## Phase 2: Data Cleaning

In [ ]:
df = df_raw.drop_duplicates().copy()
logging.info(f'Removed {len(df_raw) - len(df)} duplicates.')

col_map = {}
for col in df.columns:
    cl = col.lower()
    if cl in ['time','date','datetime']:     col_map['Datetime'] = col
    elif cl in ['latitude','lat']:           col_map['Latitude'] = col
    elif cl in ['longitude','lon','lng']:    col_map['Longitude'] = col
    elif cl == 'depth':                      col_map['Depth'] = col
    elif cl in ['mag','magnitude']:          col_map['Magnitude'] = col
for col in df.columns:
    cl = col.lower()
    if 'Datetime' not in col_map and ('time' in cl or 'date' in cl): col_map['Datetime'] = col
    elif 'Latitude' not in col_map and 'lat' in cl:                  col_map['Latitude'] = col
    elif 'Longitude' not in col_map and 'lon' in cl:                 col_map['Longitude'] = col
    elif 'Depth' not in col_map and 'depth' in cl:                   col_map['Depth'] = col
    elif 'Magnitude' not in col_map and 'mag' in cl and not any(x in cl for x in ['type','nst','source','error']):
        col_map['Magnitude'] = col

df = df.rename(columns={v: k for k, v in col_map.items()})
df['Datetime'] = pd.to_datetime(df['Datetime'], errors='coerce')
for c in ['Latitude','Longitude','Depth','Magnitude']:
    if c in df.columns: df[c] = pd.to_numeric(df[c], errors='coerce')
df = df.dropna(subset=['Datetime','Latitude','Longitude','Magnitude'])
if 'Depth' in df.columns: df['Depth'] = df['Depth'].fillna(df['Depth'].median())

print(f'Clean shape: {df.shape}')
df.head()

## Phase 3: Feature Engineering

In [ ]:
df['Year']   = df['Datetime'].dt.year
df['Month']  = df['Datetime'].dt.month
df['Day']    = df['Datetime'].dt.day
df['Hour']   = df['Datetime'].dt.hour
df['Decade'] = (df['Year'] // 10) * 10

def season(m):
    if m in [12,1,2]:    return 'Winter'
    elif m in [3,4,5]:   return 'Pre-monsoon'
    elif m in [6,7,8,9]: return 'Monsoon'
    else:                return 'Post-monsoon'

def mag_cat(m):
    if m < 4.0:   return 'Minor'
    elif m <= 5.0: return 'Moderate'
    else:          return 'Strong'

def depth_cat(d):
    if d < 70:    return 'Shallow'
    elif d <= 300: return 'Intermediate'
    else:          return 'Deep'

def get_period(y):
    if y <= 1999:  return '1990-1999'
    elif y <= 2009: return '2000-2009'
    elif y <= 2019: return '2010-2019'
    else:           return '2020-2026'

df['Season']         = df['Month'].apply(season)
df['Mag_Category']   = df['Magnitude'].apply(mag_cat)
df['Depth_Category'] = df['Depth'].apply(depth_cat)
df['Period']         = df['Year'].apply(get_period)

df.to_csv(os.path.join(OUTPUT_DIR, 'cleaned_earthquake_data.csv'), index=False)
print('Feature engineering complete. Dataset saved.')
df.head()